# **Retail Sales EDA in Python**

## Project Goal: Helping Retail Businesses Turn Data Into Decisions

This project is built for **retail business owners and e-commerce managers** in fashion, beauty, or electronics who want fast, actionable insights from their sales data.

Too often, I hear:  

> *“We have tons of sales data, but no time or tools to analyze it.  
> I don’t know which customers are most valuable, which products drive repeat purchases, or when to run promotions.  
> I need quick, clear insights to guide decisions without hiring a full data team.”*

This project explores a fictional retail dataset to uncover insights on customer behavior, product performance, and seasonal trends designed to help teams act faster, smarter, and with confidence.

## Business Impact Summary
- Identified high-value customer segments for targeted loyalty campaigns
- Revealed seasonal revenue peaks to guide inventory and promotion timing
- Uncovered bundling opportunities to lift revenue in low-margin months
- Delivered strategic recommendations aligned with cost-saving and growth

For those who want to explore the work on Github: 

👉 [Retail Sales EDA in Python](https://github.com/Wilfrida-Were/Retail-Sales-EDA-in-Python/blob/main/README.md)

I use the same dataset, to perform **[Retail Sales EDA in SQL](https://www.kaggle.com/code/wilfridawere/retail-sales-eda-in-sql)**

## 📑 Table of Contents
* [Code](#code)
  * [Data Cleaning](#data-cleaning)
  * [Distributions: Understanding the Data at a Glance](#distributions)
  * [Q1: Who are our most valuable customers?](#q1)
  * [Q2: How does customer age and gender influence purchasing behavior?](#q2)
  * [Q3: Which product categories drive the most revenue?](#q3)
  * [Q4: Monthly trends in sales and transactions](#q4)
  * [Q5: What are the patterns in purchase quantity per transaction?](#q5)
  * [Q6: How does pricing affect purchasing behavior?](#q6)
* [Key Insights](#key-insights)
* [Recommendations](#recommendations)
* [Client Scenario: Applying the Recommendations](#client-scenario)
* [Who Can Use This Project](#who-can-use-this-project)
* [Tech Stack](#tech-stack)
* [Workflow](#workflow)
* [Key Learnings](#key-learnings)
* [Let’s Connect](#lets-connect)

In [ ]:
# Install Seaborn if needed (run once)
%pip install -q seaborn

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

<a id="code"></a>
## 💻 Code

In [ ]:
# Code starts here.
# Run the cells from top to bottom for the smoothest execution.

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================
# This version works in normal Jupyter Notebook.
# Put retail_sales_dataset.csv in the same folder as this notebook.

file_name = "retail_sales_dataset.csv"
possible_paths = [
    Path.cwd() / file_name,
    Path("/kaggle/input/retail-sales-dataset") / file_name
]

dataset_path = next((p for p in possible_paths if p.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        f"'{file_name}' was not found.\n"
        "Place the CSV file in the same folder as this notebook and run this cell again."
    )

retail = pd.read_csv(dataset_path)
df = retail.copy()

print(f"Dataset loaded from: {dataset_path}")
print(f"Shape: {df.shape}")
display(df.head())

<a id="data-cleaning"></a>
## Data Cleaning 

In [ ]:
# Seaborn was already installed above, so no second installation is needed.

In [ ]:
# Data cleaning section starts here.

In [ ]:
# Preview dataset structure
print("Dataset Overview:")
df.info()  # 1000 rows and 9 columns

In [ ]:
# ============================================================
# DATA CLEANING STEP 1: FIX DATA TYPES
# ============================================================

# Convert Date from text to datetime.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Convert numeric columns safely.
numeric_columns = ["Age", "Quantity", "Price per Unit", "Total Amount"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data types after conversion:")
display(df.dtypes)

In [ ]:
# ============================================================
# DATA CLEANING STEP 2: CHECK MISSING VALUES
# ============================================================

missing_values = df.isna().sum()

print("Missing Values Check:")
display(missing_values.to_frame("Missing Values"))

if missing_values.sum() == 0:
    print("No missing values detected.")
else:
    print("Missing values found. Rows containing missing values will be removed for analysis.")

# Remove rows with missing values in columns required for analysis.
required_columns = [
    "Transaction ID", "Date", "Customer ID", "Gender",
    "Age", "Product Category", "Quantity", "Price per Unit", "Total Amount"
]
df = df.dropna(subset=required_columns).copy()

print(f"Dataset shape after cleaning: {df.shape}")

In [ ]:
# ============================================================
# DATA CLEANING STEP 3: Unique Values per Column
# ============================================================
unique_counts = df.nunique()

print("\nUnique Value Counts:")
print(unique_counts)

# Quick summary for stakeholders
print("\n✅ Interpretation:")
print("- Categorical columns with fewer unique values may represent dimensions like Gender or Product Category.")
print("- High unique counts in columns like Transaction ID or Customer ID confirm identifiers.")
print("- Helps decide which columns are categorical vs. continuous.")

From the output above:

* There are 1000 distinct Transaction IDs and 1000 distinct Customers (Customer ID)s
* Since there are only 345 unique dates, some customers likely made purchases on the same dates
* The Product categories are only *Beauty*, *Clothing* and *Electronics*

<a id="distributions"></a>
## 📊 Distributions: Understanding the Data at a Glance  

In [ ]:
# ============================================================
# KPI CALCULATIONS
# ============================================================

total_revenue = df["Total Amount"].sum()
total_transactions = df["Transaction ID"].nunique()
avg_transaction_value = df.groupby("Transaction ID")["Total Amount"].sum().mean()

min_age, max_age = df["Age"].min(), df["Age"].max()
min_price, max_price = df["Price per Unit"].min(), df["Price per Unit"].max()

transaction_amounts = df.groupby("Transaction ID")["Total Amount"].sum()
min_transaction_amount = transaction_amounts.min()
max_transaction_amount = transaction_amounts.max()

print("Dataset-Level KPIs")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Transactions: {total_transactions}")
print(f"Average Transaction Value: ${avg_transaction_value:,.2f}")
print(f"Age Range: {min_age:.0f} - {max_age:.0f}")
print(f"Price per Unit Range: ${min_price:,.2f} - ${max_price:,.2f}")
print(f"Transaction Amount Range: ${min_transaction_amount:,.2f} - ${max_transaction_amount:,.2f}")

# ============================================================
# DISTRIBUTIONS
# ============================================================

numeric_cols = ["Total Amount", "Quantity", "Age", "Price per Unit"]
categorical_cols = ["Gender", "Product Category"]

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=20, edgecolor="black")
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

gender_counts = df["Gender"].value_counts()
axes[4].bar(gender_counts.index, gender_counts.values, edgecolor="black")
axes[4].set_title("Gender Distribution")
axes[4].set_ylabel("Count")

product_counts = df["Product Category"].value_counts()
axes[5].bar(product_counts.index, product_counts.values, edgecolor="black")
axes[5].set_title("Product Category Distribution")
axes[5].set_ylabel("Count")
axes[5].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

<a id="q1"></a>
## Q1: Who are our most valuable customers?

In [ ]:
# ============================================================
# Q1: WHO ARE OUR MOST VALUABLE CUSTOMERS?
# ============================================================

customer_summary = df.groupby("Customer ID").agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique"),
    Avg_Transaction_Value=("Total Amount", "mean"),
    Total_Quantity=("Quantity", "sum"),
    Age=("Age", "first"),
    Gender=("Gender", "first"),
    Top_Product_Category=("Product Category", lambda x: x.mode().iat[0])
).reset_index()

top_customers = customer_summary.sort_values(
    "Total_Revenue", ascending=False
).head(10)

print("Table 1: Top 10 Customers by Revenue")
display(top_customers)

plot_data = top_customers.sort_values("Total_Revenue")

plt.figure(figsize=(10, 6))
plt.barh(plot_data["Customer ID"].astype(str), plot_data["Total_Revenue"])
plt.title("Top 10 Customers by Total Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Customer ID")

for i, value in enumerate(plot_data["Total_Revenue"]):
    plt.text(value, i, f" ${value:,.0f}", va="center")

plt.tight_layout()
plt.show()

<a id="q2"></a>
## Q2: How does customer age and gender influence purchasing behavior?

In [ ]:
# ============================================================
# Q2: HOW DO AGE AND GENDER INFLUENCE PURCHASING BEHAVIOR?
# ============================================================

# 2A. Spending by Age
age_summary = df.groupby("Age").agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique"),
    Avg_Transaction_Value=("Total Amount", "mean"),
    Num_Customers=("Customer ID", "nunique")
).reset_index()

top_age_summary = age_summary.sort_values(
    "Total_Revenue", ascending=False
).head(10)

print("Table 2A: Top 10 Ages by Total Revenue")
display(top_age_summary)

plt.figure(figsize=(10, 6))
plt.bar(top_age_summary["Age"].astype(int).astype(str), top_age_summary["Total_Revenue"])
plt.title("Top 10 Ages by Total Revenue")
plt.xlabel("Age")
plt.ylabel("Total Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 2B. Spending by Gender
gender_summary = df.groupby("Gender").agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique"),
    Avg_Transaction_Value=("Total Amount", "mean"),
    Num_Customers=("Customer ID", "nunique")
).reset_index()

print("Table 2B: Spending Summary by Gender")
display(gender_summary)

plt.figure(figsize=(7, 5))
plt.bar(gender_summary["Gender"], gender_summary["Total_Revenue"])
plt.title("Total Revenue by Gender")
plt.xlabel("Gender")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

# 2C. Spending by Age and Gender
age_gender_summary = df.groupby(["Age", "Gender"]).agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique"),
    Avg_Transaction_Value=("Total Amount", "mean"),
    Num_Customers=("Customer ID", "nunique")
).reset_index()

top_age_gender_table = age_gender_summary.sort_values(
    "Total_Revenue", ascending=False
).head(10)

print("Table 2C: Top Age & Gender Combinations by Revenue")
display(top_age_gender_table)

pivot_age_gender = age_gender_summary.pivot_table(
    index="Age", columns="Gender", values="Total_Revenue", aggfunc="sum", fill_value=0
)

top_ages = pivot_age_gender.sum(axis=1).nlargest(10).index
pivot_age_gender.loc[top_ages].sort_index().plot(
    kind="bar", stacked=True, figsize=(11, 6)
)

plt.title("Top 10 Ages by Total Revenue, Split by Gender")
plt.xlabel("Age")
plt.ylabel("Total Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

<a id="q3"></a>
## Q3: Which product categories drive the most revenue?

In [ ]:
# ============================================================
# Q3: WHICH PRODUCT CATEGORIES DRIVE THE MOST REVENUE?
# ============================================================

# 3A. Revenue by Product Category
product_summary = df.groupby("Product Category").agg(
    Total_Revenue=("Total Amount", "sum"),
    Number_of_Customers=("Customer ID", "nunique"),
    Avg_Transaction_Value=("Total Amount", "mean")
).reset_index().sort_values("Total_Revenue", ascending=False)

print("Table 3A: Product Category Revenue Summary")
display(product_summary)

plt.figure(figsize=(7, 7))
plt.pie(
    product_summary["Total_Revenue"],
    labels=product_summary["Product Category"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Revenue Contribution by Product Category")
plt.show()

# 3B. Revenue by Product Category and Gender
product_gender_summary = df.groupby(
    ["Product Category", "Gender"]
).agg(
    Total_Revenue=("Total Amount", "sum"),
    Number_of_Customers=("Customer ID", "nunique"),
    Avg_Transaction_Value=("Total Amount", "mean")
).reset_index()

print("Table 3B: Revenue by Product Category & Gender")
display(product_gender_summary.sort_values("Total_Revenue", ascending=False))

pivot_product_gender = product_gender_summary.pivot_table(
    index="Product Category",
    columns="Gender",
    values="Total_Revenue",
    aggfunc="sum",
    fill_value=0
)

pivot_product_gender.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.title("Revenue by Product Category and Gender")
plt.xlabel("Product Category")
plt.ylabel("Total Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 3C. Top 3 Age/Gender combinations for each product
age_gender_product_summary = df.groupby(
    ["Product Category", "Age", "Gender"]
).agg(
    Total_Revenue=("Total Amount", "sum"),
    Number_of_Customers=("Customer ID", "nunique")
).reset_index()

top3_age_gender_per_product = (
    age_gender_product_summary
    .sort_values(["Product Category", "Total_Revenue"], ascending=[True, False])
    .groupby("Product Category", group_keys=False)
    .head(3)
)

print("Table 3C: Top 3 Age & Gender Combinations per Product")
display(top3_age_gender_per_product)

for product in top3_age_gender_per_product["Product Category"].unique():
    data = top3_age_gender_per_product[
        top3_age_gender_per_product["Product Category"] == product
    ].copy()

    labels = data["Age"].astype(int).astype(str) + " / " + data["Gender"]

    plt.figure(figsize=(7, 5))
    plt.bar(labels, data["Total_Revenue"])
    plt.title(f"{product}: Top 3 Age/Gender Groups by Revenue")
    plt.xlabel("Age / Gender")
    plt.ylabel("Revenue")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

<a id="q4"></a>
## Q4: Monthly trends in sales and transactions

In [ ]:
# ============================================================
# Q4: MONTHLY TRENDS IN SALES AND TRANSACTIONS
# ============================================================

df["Month"] = df["Date"].dt.month

monthly_sales = df.groupby("Month").agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique")
).reset_index()

monthly_sales["Avg_Transaction_Value"] = (
    monthly_sales["Total_Revenue"] / monthly_sales["Total_Transactions"]
)

print("Table 4: Monthly Sales Summary")
display(monthly_sales)

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=monthly_sales,
    x="Month",
    y="Total_Revenue",
    marker="o"
)
plt.title("Monthly Total Revenue")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(monthly_sales["Month"])
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(
    data=monthly_sales,
    x="Month",
    y="Total_Transactions",
    marker="o"
)
plt.title("Monthly Number of Transactions")
plt.xlabel("Month")
plt.ylabel("Total Transactions")
plt.xticks(monthly_sales["Month"])
plt.tight_layout()
plt.show()

<a id="q5"></a>
## Q5: What are the patterns in purchase quantity per transaction?

In [ ]:
# ============================================================
# Q5: PATTERNS IN PURCHASE QUANTITY PER TRANSACTION
# ============================================================

avg_quantity = df["Quantity"].mean()
print(f"Average Quantity per Transaction: {avg_quantity:.2f}")

avg_quantity_by_product = (
    df.groupby("Product Category")["Quantity"]
    .mean()
    .reset_index(name="Avg_Quantity")
)

print("Table 5A: Average Quantity by Product Category")
display(avg_quantity_by_product)

avg_qty_month_product = df.groupby(
    ["Month", "Product Category"]
).agg(
    Avg_Quantity=("Quantity", "mean")
).reset_index()

avg_qty_pivot = avg_qty_month_product.pivot_table(
    index="Month",
    columns="Product Category",
    values="Avg_Quantity",
    aggfunc="mean"
).round(2)

print("Table 5B: Average Quantity by Product and Month")
display(avg_qty_pivot)

plt.figure(figsize=(11, 6))

for product in avg_qty_month_product["Product Category"].unique():
    product_data = avg_qty_month_product[
        avg_qty_month_product["Product Category"] == product
    ]
    plt.plot(
        product_data["Month"],
        product_data["Avg_Quantity"],
        marker="o",
        label=product
    )

plt.title("Average Quantity by Product Category per Month")
plt.xlabel("Month")
plt.ylabel("Average Quantity")
plt.xticks(range(1, 13))
plt.legend(title="Product Category")
plt.tight_layout()
plt.show()

<a id="q6"></a>
## Q6: How does pricing affect purchasing behavior?

In [ ]:
# ============================================================
# Q6: HOW DOES PRICING AFFECT PURCHASING BEHAVIOR?
# ============================================================

# 6A. Pricing Summary
price_summary = df.groupby("Price per Unit").agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique"),
    Avg_Quantity=("Quantity", "mean")
).reset_index()

print("Table 6A: Pricing Summary")
display(price_summary)

# 6B. Revenue by Price and Product Category
price_product_summary = df.groupby(
    ["Price per Unit", "Product Category"]
).agg(
    Total_Revenue=("Total Amount", "sum"),
    Total_Transactions=("Transaction ID", "nunique"),
    Avg_Quantity=("Quantity", "mean")
).reset_index()

revenue_pivot = price_product_summary.pivot_table(
    index="Price per Unit",
    columns="Product Category",
    values="Total_Revenue",
    aggfunc="sum",
    fill_value=0
)

print("Table 6B: Total Revenue per Product by Price")
display(revenue_pivot)

# Optional visualization
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="Price per Unit",
    y="Quantity",
    hue="Product Category"
)
plt.title("Price per Unit vs Quantity Purchased")
plt.xlabel("Price per Unit")
plt.ylabel("Quantity")
plt.tight_layout()
plt.show()

<a id="key-insights"></a>
## 🔑 Key Insights

### Q1: Who are our most valuable customers?

**Key Insights:**
- **High Revenue Individuals:** Top 10 customers each contributed USD 2,000, far above the average transaction value (USD 456).
- **Product Preference:** Electronics and Clothing appear most frequently among top customers, suggesting repeat or high-value transactions.
- **Age & Gender Distribution:** Top customers are aged 22–62, with both males and females represented.

### Q2: How does customer age and gender influence purchasing behavior?

**Key Insights:**
- **Age Trends:** Middle-aged customers (34–51) drive revenue via consistent transactions.
- **High-Value Buyers:** Younger customers (19–37) make fewer but larger purchases.
- **Gender Patterns:** Females slightly outspend males (USD 232,840 vs USD 223,160), with similar average transaction values (~$456).
- **Top Age-Gender Combinations:** Female 34–26 and Male 46–51 segments show highest total revenue.

### Q3: Which product categories drive the most revenue?

**Key Insights:**
- Electronics generate the highest revenue overall, slightly ahead of Clothing.
- Beauty has highest average transaction value (USD 467).
- Female buyers dominate Clothing and Beauty; males favor Electronics.
- Older males (46–63) drive high-revenue Electronics and Beauty; women 25–64 drive Clothing revenue.
  
### Q4: Monthly trends in sales and transactions

**Key Insights:**
- **Peak months:** May (USD 53,150) and October (USD 46,580).
- **Low months:** September (USD 23,620) and March (USD 28,990).
- **Average transaction value peaks:** February (USD 518), July (USD 493), December (USD 491).

### Q5: Patterns in purchase quantity per transaction

**Key Insights:**
- Average quantity per transaction: 2.51.
- Clothing purchases slightly higher quantity; Electronics lowest.
- Peaks occur for Clothing in months 3 & 9; Electronics in Feb, June, Oct.

### Q6: How does pricing affect purchasing behavior?

**Key Insights:**
- High-priced products (USD 300–USD 500) generate most revenue despite fewer transactions.
- Low-priced products (USD 25–USD 50) have more transactions but less total revenue.
- Average quantity per transaction remains stable (~ 2.4–2.6), showing price does not deter unit purchases.

---

<a id="recommendations"></a>
## ✅ Recommendations
These recommendations are based on a full analysis of customer behavior, product performance, seasonal trends, and pricing sensitivity. Each one is **specific, time-bound**, and **ready to implement** using lightweight tools and workflows.

> **🛠️ How to Use This Section:**  
> As a retail manager, you don’t need to implement everything at once.  
> Use this section as a **decision support guide**—pick the strategies that fit your current goals, season, and customer base.  
> Whether you're planning a campaign, adjusting inventory, or refining pricing, these insights give you **clarity and direction** so you can act faster, smarter, and with confidence.


### 🧍‍♀️ Customer Loyalty & Segmentation

- **Launch a VIP program** for customers who’ve spent over $1,500 in the past 6 months  
  → Offer early access to Electronics (Feb, Oct) and exclusive Clothing bundles (March, Sept)

- **Send personalized offers** to top 10% customers  
  → Use email/SMS tools to recommend products based on past purchases and seasonal trends

- **Segment campaigns by age-gender clusters**  
  → Female 34–46: Clothing & Beauty  
  → Male 46–63: Electronics + Beauty  
  → Younger buyers (19–37): High-ticket Electronics with flexible payment options


### 🛍️ Product Bundling & Promotions

- **Bundle low-margin Clothing with high-margin Beauty**  
  → Target March & September with “Back-to-style” kits (e.g., jackets + skincare)

- **Upsell Electronics accessories** during high-quantity months  
  → February, June, October: Bundle smartwatches with chargers, bands, or cases

- **Create seasonal bundles based on quantity trends**  
  → March & September: Clothing bundles (3+ items)  
  → February: Electronics multi-unit offers (“Buy 2, get 10% off”)


### 📅 Seasonal Planning & Inventory

- **Stock up for May and October**, your highest revenue months  
  → Prioritize Electronics and Clothing; launch campaigns 2–3 weeks early

- **Reposition September as a volume-driven month**  
  → Clothing sells most but earns least—use bundling and upselling to lift revenue

- **Use December for gift bundles**, not premium Electronics  
  → Focus on curated Beauty and Clothing sets with loyalty perks


### 💰 Pricing Strategy

- **Promote high-ticket Electronics in February**  
  → Quantity per transaction peaks—ideal for bundling and tiered pricing

- **Use low-priced items ($25–$50) as entry points**  
  → Drive volume, then upsell via bundles or loyalty rewards

- **Avoid premium Electronics pushes in December**  
  → Quantity is low—focus on Beauty and Clothing upsells instead


### ⚙️ Operational Efficiency

- **Automate segmentation and outreach**  
  → Use n8n or Zapier to trigger campaigns based on spend, product interest, or season

- **Monitor age-gender-product intersections**  
  → Build dashboards to refine targeting and inventory decisions in real time

---

<a id="client-scenario"></a>
## 🧩 Client Scenario: Applying the Recommendations  
Let’s say you manage a mid-sized fashion and electronics store with seasonal promotions and a growing loyalty base. Here’s how you could apply the recommendations in real life:


### 🎯 Goal: Boost February Sales with High-Margin Electronics

- Identify top male customers aged 46–63  
- Launch a “Smart Tech Bundle” campaign: smartwatch + charger + case  
- Offer tiered discounts: Buy 2, get 10% off; Buy 3, get 15% off  
- Send personalized SMS offers to high spenders

**Why it works:**  
February has the highest Electronics quantity per transaction. Older males favor high-ticket items. Bundling lifts revenue without increasing acquisition costs.



### 🧥 Goal: Lift September Revenue Despite Low Margins

- Bundle popular Clothing items (jackets, tops) with Beauty accessories  
- Promote “Back-to-Style” kits via email and Instagram  
- Target female customers aged 34–46 with curated sets  
- Offer free shipping for bundles over $100

**Why it works:**  
September has high Clothing volume but low revenue. Bundling with Beauty increases average transaction value. Targeting high-revenue segments improves campaign ROI.



### 📦 Goal: Prepare Inventory for May and October Peaks

- Analyze top-selling Electronics and Clothing SKUs from previous May and October  
- Increase stock 2–3 weeks ahead of each peak  
- Schedule loyalty emails with early access offers  
- Use dashboards to monitor age-gender-product trends in real time

**Why it works:**  
May and October are revenue peak months. Early access drives urgency and repeat purchases. Inventory alignment reduces stockouts and overstocking.

<a id="who-can-use-this-project"></a>
## 👥 Who Can Use This Project?

**1. Business Leaders / Managers**  
- Make fast, data-driven decisions on customers, products, and timing.

**2. Marketing & Sales Teams**  
- Plan targeted campaigns and seasonal promotions with confidence.

**3. Data Analysts / Data Enthusiasts**  
- Practice real-world EDA and build portfolio-ready insights.

**4. Learners / Students**  
- Learn how data drives business strategy and clear communication.

<a id="tech-stack"></a>
## ⚙️ Tech Stack
- **Python (Pandas, Matplotlib, Seaborn)** – Data cleaning, preprocessing, and exploratory analysis  
- **Kaggle Notebook** – End-to-end workflow combining code, analysis, and documentation 

<a id="workflow"></a>
## 🔄 Workflow
1. **Raw Data** → [Kaggle Dataset: Retail Sales Dataset](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset/data)  
2. **Data Cleaning & Preprocessing** → Performed in Python (Pandas) to handle missing values, fix formatting, and prepare data for analysis.  
3. **Exploratory Data Analysis (EDA)** → Conducted in Python using Pandas for aggregations and Matplotlib/Seaborn for identifying trends and patterns.  
4. **Visualization & Storytelling** → Built charts with Matplotlib and Seaborn to present insights in a clear, business-focused manner.  

<a id="key-learnings"></a>
## 📌 Key Learnings
- Handling missing and inconsistent data using **Pandas**.  
- Structuring cleaned datasets for **efficient analysis and aggregation**.  
- Applying **EDA techniques** to uncover trends and patterns.  
- Creating **visualizations with Matplotlib & Seaborn** to communicate business insights.  

<a id="lets-connect"></a>
## 🔗 Let’s Connect

Feel free to connect, follow, or support:  

[![LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-blue?style=flat&logo=linkedin)](https://linkedin.com/in/wilfridawere/)  

[![Twitter](https://img.shields.io/badge/X-Follow-black?style=flat&logo=twitter)](https://x.com/wilfridawere)  

[![Website](https://img.shields.io/badge/Website-Visit-orange?style=flat&logo=google-chrome)](https://www.wilfridawere.com/)  

[![Kaggle](https://img.shields.io/badge/Kaggle-Follow-blue?style=flat&logo=kaggle)](https://kaggle.com/wilfridawere)  

[![GitHub](https://img.shields.io/badge/GitHub-Projects-black?style=flat&logo=github)](https://github.com/Wilfrida-Were)  